# The `%%Makemymodel` Jupyter Magic

The `%%Makemymodel` cell magic lets you write **model specifications directly in notebook cells** using Markdown-style text mixed with ModelFlow DSL equations. Under the hood it parses the cell, feeds everything to `Makemodel`, and pushes the resulting object into the notebook namespace — ready to use.

This notebook explains how it works and walks through all the options.

In [1]:
import modeljupytermagic          # registers the magics
from modelconstruct_estimation import Makemodel

## 1. Basic syntax

The first line after `%%Makemymodel` carries the **model name** and optional **flags**:

```
%%Makemymodel <name> [option1] [option2=value] ...
```

The rest of the cell is **Markdown text** where lines starting with `>` are treated as ModelFlow DSL equations. Everything else is documentation that gets rendered in the notebook.

After execution the `Makemodel` instance is available as a variable with the given `<name>`.

### Minimal example

The cell below creates a `Makemodel` object called `simple` and renders the Markdown text in the notebook.

In [2]:
%%Makemymodel simple
A simple two-equation model
> y = c + i
> c = 0.8 * y(-1)

A simple two-equation model
```text
> y = c + i
> c = 0.8 * y(-1)
```

The variable `simple` is now a `Makemodel` instance in the notebook namespace. We can inspect it:

In [3]:
# Show the expanded equations
simple.show

FRML <> Y = C+I $
FRML <> C = 0.8*Y(-1) $


In [4]:
# Draw the causal graph
simple.draw

## 2. Available options

Options are placed on the first line after the model name. They follow the pattern `key` (boolean flag) or `key=value`.

| Option | Default | Effect |
|---|---|---|
| `show` | off | Print the expanded equations after creation |
| `draw` | off | Display the causal dependency graph |
| `display` | off | Verbose mode: render Markdown **and** print equations + segment info |
| `render_est` | on | Render the Markdown text with estimation results included |
| `render` | on | Render the plain Markdown text (only consulted when `render_est=0`) |
| `render_list` | on | Include list definitions when rendering (set to `0` to hide them) |
| `latex` | off | Generate a LaTeX PDF of the model specification |
| `spec` | `markdown` | Rendering format (`markdown` by default) |
| `segment` | off | Segmented model building (see Section 4) |
| `replacements` | none | Dimension placeholders, e.g. `replacements=('__d','__{banks}')` |
| `funks` | none | User-defined functions to pass to `Makemodel` |
| `estimator` (alias `est`) | none | Default estimator for equations tagged `<est>`/`<estimator>` without a value (see Section 6) |
| `input_df` | none | `DataFrame` used when tagged equations are estimated |
| `smpl` | none | Model-wide default estimation sample, e.g. `smpl="(2012, 2019)"` |

Values that are not literals are looked up as names in the notebook namespace, so `input_df=npl` or `estimator=ls` refer to notebook variables. Boolean options can be turned off explicitly with `option=0` or `option=False`.

### Example: `show` and `draw`

In [5]:
%%Makemymodel demo show draw
A model with tags
><exo,add> y = c + i
> c = 0.8 * y(-1)

A model with tags
```text
><exo,add> y = c + i
> c = 0.8 * y(-1)
```

FRML <EXO,ADD> Y = (C+I + Y_A)* (1-Y_D)+ Y_X*Y_D  $
FRML <> C = 0.8*Y(-1) $

FRML <CALC_ADD_FACTOR>  Y_A = (Y) - (C+I) $


### Example: suppressing rendering with `render=0`

In [6]:
%%Makemymodel quiet render=0 show
This text will NOT appear in the output
> a = b + c

This text will NOT appear in the output
```text
> a = b + c
```

FRML <> A = B+C $


## 3. Using lists and `doable` inside the magic

The cell content is passed directly to `Makemodel`, so all DSL features — lists, `do...enddo`, `doable`, `sum()`, tags — work exactly as they would in a normal `Makemodel()` call.

In [7]:
%%Makemymodel bank_model show
## Credit risk model

Define the banks and portfolios:

>list banks  = banks : Danske Nordea 
>list ports  = ports : Mortgage Corporate 

Calculate expected loss per bank and portfolio, then aggregate:

>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}

## Credit risk model

Define the banks and portfolios:

```text
>list banks  = banks : Danske Nordea 
>list ports  = ports : Mortgage Corporate 
```

Calculate expected loss per bank and portfolio, then aggregate:

```text
>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}
```

FRML <SUM=TOTAL> EL__DANSKE__MORTGAGE = EXPOSURE__DANSKE__MORTGAGE*PD__DANSKE__MORTGAGE $
FRML <SUM=TOTAL> EL__DANSKE__CORPORATE = EXPOSURE__DANSKE__CORPORATE*PD__DANSKE__CORPORATE $
FRML <SUM=TOTAL> EL__NORDEA__MORTGAGE = EXPOSURE__NORDEA__MORTGAGE*PD__NORDEA__MORTGAGE $
FRML <SUM=TOTAL> EL__NORDEA__CORPORATE = EXPOSURE__NORDEA__CORPORATE*PD__NORDEA__CORPORATE $
FRML <SUM=TOTAL> EL__TOTAL = ((EL__DANSKE__MORTGAGE+EL__DANSKE__CORPORATE)+(EL__NORDEA__MORTGAGE+EL__NORDEA__CORPORATE)) $


## 4. Segmented models

Large models can be split across **multiple cells** using the `segment=<name>` option. This is one of the most powerful features of the magic.

**How it works:**

1. Each cell with `segment=<name>` stores its content in a dictionary called `<modelname>_dict`.
2. Segments whose name starts with `list` or `text` are treated as definitions/documentation — they are stored but only rendered, not yet compiled.
3. When a non-list/non-text segment is encountered, the magic combines **all list segments** with the **current cell** and builds the `Makemodel` model.
4. A final call **without** `segment` combines **all** stored segments into the complete model.

This lets you build a model incrementally, cell by cell, with explanatory text between sections.

A practical naming convention: call the `Makemodel` instance `M<name>` (e.g. `Mtoymodel`) so the plain name stays free for the solvable model instance, extracted after the combining call with `toymodel = Mtoymodel.mmodel` — see Section 6.

In [8]:
%%Makemymodel stress segment=listtests
## List definitions
>list banks  = banks : Alpha Beta 
>list ports  = ports : Retail SME 

## List definitions
```text
>list banks  = banks : Alpha Beta 
>list ports  = ports : Retail SME 
```

In [9]:
%%Makemymodel stress segment=pd_block show
## PD dynamics
Probability of default follows an AR(1) process driven by GDP growth:
>doable <> pd__{banks}__{ports} = 0.02 + 0.8 * pd__{banks}__{ports}(-1) + 0.1 * gdp_growth

## List definitions
```text
>list banks  = banks : Alpha Beta 
>list ports  = ports : Retail SME 
```
## PD dynamics
Probability of default follows an AR(1) process driven by GDP growth:
```text
>doable <> pd__{banks}__{ports} = 0.02 + 0.8 * pd__{banks}__{ports}(-1) + 0.1 * gdp_growth
```

FRML <> PD__ALPHA__RETAIL = 0.02+0.8*PD__ALPHA__RETAIL(-1)+0.1*GDP_GROWTH $
FRML <> PD__ALPHA__SME = 0.02+0.8*PD__ALPHA__SME(-1)+0.1*GDP_GROWTH $
FRML <> PD__BETA__RETAIL = 0.02+0.8*PD__BETA__RETAIL(-1)+0.1*GDP_GROWTH $
FRML <> PD__BETA__SME = 0.02+0.8*PD__BETA__SME(-1)+0.1*GDP_GROWTH $


In [10]:
%%Makemymodel stress segment=loss_block show
## Expected loss
>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}

## List definitions
```text
>list banks  = banks : Alpha Beta 
>list ports  = ports : Retail SME 
```
## Expected loss
```text
>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}
```

FRML <SUM=TOTAL> EL__ALPHA__RETAIL = EXPOSURE__ALPHA__RETAIL*PD__ALPHA__RETAIL $
FRML <SUM=TOTAL> EL__ALPHA__SME = EXPOSURE__ALPHA__SME*PD__ALPHA__SME $
FRML <SUM=TOTAL> EL__BETA__RETAIL = EXPOSURE__BETA__RETAIL*PD__BETA__RETAIL $
FRML <SUM=TOTAL> EL__BETA__SME = EXPOSURE__BETA__SME*PD__BETA__SME $
FRML <SUM=TOTAL> EL__TOTAL = ((EL__ALPHA__RETAIL+EL__ALPHA__SME)+(EL__BETA__RETAIL+EL__BETA__SME)) $


Now combine all segments into the full model by calling `%%Makemymodel` (or `%Makemymodel` as a line magic) **without** the `segment` option:

In [11]:
%%Makemymodel stress show draw
## Full stress test model
Combining all segments into one model.

## PD dynamics
Probability of default follows an AR(1) process driven by GDP growth:
```text
>doable <> pd__{banks}__{ports} = 0.02 + 0.8 * pd__{banks}__{ports}(-1) + 0.1 * gdp_growth
```

## Expected loss
```text
>doable <sum=total> el__{banks}__{ports} = exposure__{banks}__{ports} * pd__{banks}__{ports}
```
## Full stress test model
Combining all segments into one model.

## List definitions
```text
>list banks  = banks : Alpha Beta 
>list ports  = ports : Retail SME 
```

FRML <> PD__ALPHA__RETAIL = 0.02+0.8*PD__ALPHA__RETAIL(-1)+0.1*GDP_GROWTH $
FRML <> PD__ALPHA__SME = 0.02+0.8*PD__ALPHA__SME(-1)+0.1*GDP_GROWTH $
FRML <> PD__BETA__RETAIL = 0.02+0.8*PD__BETA__RETAIL(-1)+0.1*GDP_GROWTH $
FRML <> PD__BETA__SME = 0.02+0.8*PD__BETA__SME(-1)+0.1*GDP_GROWTH $
FRML <SUM=TOTAL> EL__ALPHA__RETAIL = EXPOSURE__ALPHA__RETAIL*PD__ALPHA__RETAIL $
FRML <SUM=TOTAL> EL__ALPHA__SME = EXPOSURE__ALPHA__SME*PD__ALPHA__SME $
FRML <SUM=TOTAL> EL__BETA__RETAIL = EXPOSURE__BETA__RETAIL*PD__BETA__RETAIL $
FRML <SUM=TOTAL> EL__BETA__SME = EXPOSURE__BETA__SME*PD__BETA__SME $
FRML <SUM=TOTAL> EL__TOTAL = ((EL__ALPHA__RETAIL+EL__ALPHA__SME)+(EL__BETA__RETAIL+EL__BETA__SME)) $


## 5. Using `replacements=` for dimension placeholders

When variable names have many dimensions, you can define a replacement tuple in the notebook and pass it by name:

In [12]:
repl = ('__d', '__{banks}__{ports}')

In [13]:
%%Makemymodel compact show replacements=repl
>list banks = banks : A B 
>list ports = ports : X Y 
Compact notation using `__d` as placeholder:
>doable <> loss__d = exp__d * pd__d * lgd__d

```text
>list banks = banks : A B 
>list ports = ports : X Y 
```
Compact notation using `__d` as placeholder:
```text
>doable <> loss__d = exp__d * pd__d * lgd__d
```

FRML <> LOSS__A__X = EXP__A__X*PD__A__X*LGD__A__X $
FRML <> LOSS__A__Y = EXP__A__Y*PD__A__Y*LGD__A__Y $
FRML <> LOSS__B__X = EXP__B__X*PD__B__X*LGD__B__X $
FRML <> LOSS__B__Y = EXP__B__Y*PD__B__Y*LGD__B__Y $


## 6. Estimation inside the magic

Equations tagged with `<estimator=...>` are **estimated while the model is built** — the *Estimation* chapter tells the full story. The estimator name is looked up in the notebook namespace, so a factory created with `.with_defaults()` can be referenced directly in the tag. The magic renders the estimation results (coefficients, t-statistics, fit measures) below the cell; suppress this with `render_est=0`.

In [14]:
import numpy as np
import pandas as pd
from modelestimator_new import Estimate_nls

# A small synthetic dataset: CON is (noisy) 60 percent of Y
years  = list(range(2000, 2021))
y      = pd.Series([1000 * 1.03**t for t in range(len(years))], index=years)
noise  = np.exp(np.random.default_rng(42).normal(0, 0.01, len(years)))
df_est = pd.DataFrame({'Y': y, 'CON': 0.6 * y * noise})

ls = Estimate_nls.with_defaults(smpl=(2001, 2020), input_df=df_est,
                                param_names=['lambda', 'beta_longterm', 'beta_shortterm'])

In [15]:
%%Makemymodel Mest segment=consumption
## Private consumption
><stoc,estimator=ls> dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)

Item,Value
Estimator,ls
Effective sample,"(np.int64(2001), np.int64(2020))"
Original equation,DLOG(CON) = -LAMBDA(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS(BETA_LONGTERM(1)))) + BETA_SHORTTERM(10)DLOG(Y)
Estimated equation,DLOG(CON) = -(1.0885233961)(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.5995680166)))) + (1.0061077338)DLOG(Y)
success,True
message,ftol termination condition is satisfied.
chisqr,0.001417132888
redchi,8.33607581e-05
aic,-185.0970363
bic,-182.1098395


A useful naming convention for estimated, segmented models: name the `Makemodel` instance `M<name>` — here `Mest` — and extract the solvable `modelclass.model` instance from the combining call with `.mmodel`:

In [16]:
%Makemymodel Mest render=False render_est=False
est = Mest.mmodel

The combined instance offers, among other things:

- `.render_est` — render the text plus estimation tables
- `.tvalues` — t-statistics for all estimated equations
- `.estimation_report(open_file=True)` — HTML report with actual-versus-fitted plots
- `.report().show()` — full model documentation with collapsible estimation panels
- `.init_addfactors(df, start, end, check=True)` — add factors so the model reproduces the data
- `.show<attribute>` — print any attribute nicely; an unknown name (e.g. `Mest.showclean`) triggers an error listing all the available show-properties

## 7. Tags inside the magic

All `Makemodel` tags work inside the magic cells. Here is a quick reference:

- `<exo>` — allow fixing the variable to an exogenous value
- `<add>` — generate add-factor adjustment variables
- `<fit>` — compute the original fitted value alongside the adjusted one
- `<sum=label>` — inside `doable`, also generate an aggregation equation
- `<add_suffix=__J>` — change the default add-factor suffix from `_A`
- `<stoc>` — mark the equation as behavioural: implies `<exo>` and `<add>`
- `<estimator=name>` / `<est=name>` — estimate the equation with the named estimator (see Section 6)
- `<smpl=start end>` — equation-local estimation sample
- `<caption='text'>` — caption used in estimation output and reports
- `<drop>` — leave the equation out of the emitted model (the text is kept)

Tags are placed between `><` and `>` at the start of a formula line:

In [17]:
%%Makemymodel tagged show
A fully decorated equation:
><exo,add,fit> consumption = 0.8 * income

A fully decorated equation:
```text
><exo,add,fit> consumption = 0.8 * income
```

FRML <EXO,ADD,FIT> CONSUMPTION = (0.8*INCOME + CONSUMPTION_A)* (1-CONSUMPTION_D)+ CONSUMPTION_X*CONSUMPTION_D  $
FRML <FIT>  CONSUMPTION_FITTED = 0.8*INCOME $ 
FRML <CALC_ADD_FACTOR>  CONSUMPTION_A = (CONSUMPTION) - (0.8*INCOME) $


## 8. Generating a LaTeX PDF

With the `latex` option, the magic converts the cell's Markdown text (headings, tables, bullet lists, equation lines) into LaTeX and compiles a PDF. This requires a LaTeX installation on the system.

```python
%%Makemymodel mymodel latex
# My Model Documentation
> y = c + i
```

## 9. How it works internally

The magic is implemented in `modeljupytermagic.py`. Here is the flow:

1. **Parse the first line** — `get_options(line)` splits it into the model name and a dictionary of options. Values that are not literals are resolved as names in the notebook namespace (`_resolve_option`).
2. **Handle segments** — if `segment=` is present, the cell text is stored in `<name>_dict[segment]`. List/text segments are rendered and returned early. Other segments combine list definitions with the current cell.
3. **Build the model** — the combined text is passed to `Makemodel(model_text, replacements=..., funks=..., estimator=..., input_df=..., smpl=...)`. This does all DSL expansion: lists, loops, doable, sums, tags. The notebook namespace is passed as `estimator_namespace`, so `<estimator=name>` tags resolve to notebook variables and equations so tagged are estimated on the spot.
4. **Push to namespace** — the resulting `Makemodel` instance is injected into the notebook's user namespace via `ip.push()`.
5. **Render output** — depending on options, the magic renders Markdown with estimation results (`render_est`, the default) or plain (`render`), prints expanded equations (`.show`), draws the dependency graph (`.draw`), or generates LaTeX.

The key insight is that the magic is a **thin wrapper** — all the heavy lifting is done by `Makemodel`. The magic simply provides a convenient notebook-native interface for defining and documenting models in one place.

## 10. Quick reference

```text
%%Makemymodel <name> [show] [draw] [display] [render=0] [render_est=0] [render_list=0]
              [latex] [segment=<seg>] [replacements=<var>] [funks=<var>]
              [estimator=<var>] [input_df=<var>] [smpl="(start, end)"]
<Markdown text with > prefixed DSL equations>
```

After execution: the variable `<name>` holds the `Makemodel` instance with all the usual properties — `.show`, `.draw`, `.render`, `.render_est`, `.tvalues`, `.estimation_report(...)`, `.report().show()`, `.init_addfactors(...)`, the `show<attribute>` printers, and `.mmodel` for the solvable `modelclass.model` instance.

For segmented, estimated models the recommended pattern is:

```text
%%Makemymodel Mtoymodel segment=consumption      # one cell per segment
...
%Makemymodel Mtoymodel render=False render_est=False   # combine
toymodel = Mtoymodel.mmodel                            # extract the solvable model
```